# Nemo Synapse/NVIDIA OO Agents

### Agents as objects in a software runtime

## Motivation: Stop Reinventing Software

Agent frameworks today reinvent control flow, parallelism, and state management **from scratch**.

Tool definition, library design, orchestration — these are **software problems**, not AI problems.

Agent work should focus on **agents**, not on reimplementing what programming languages already solved.

## Motivation: Recast "Agentic Stuff" as APIs

Tool calling, context management, history, orchestration → **all become APIs**

Usable by developers **or** agents — same interface, same code.

Agents and developers just write code. That's it.

**An agent is a Python object. Its methods are its capabilities. It writes code to do work. The runtime manages everything else.**

## Nemo Synapse/NVIDIA OO Agents: What Does Success Look Like?

1. **Happy developers.** Just edit your agent source file. Write software the way you're used to. No bespoke toolkits.

    - **Reusable agent software.** Design software for agents and distribute as libraries. Think "Skills" but full SW libraries, not just prompts with CLI tools.

    - **Reusable agents.** Once an agent gets good at a job, it should be reusable — the same way we have reusable software today. Import and invoke.

2. **Runtime-assisted Reasoning** We believe that models trained in this runtime will be powerful problem solvers as they can use the runtime to assist their reasoning process, manage their own context window, write and invoke subagents. 

3. **Trace-guided optimization.** When agents are just source files, we can use coding agents to rewrite the source based on feedback from traces. Rewriting any part of the agent (tools, orchestration, context engineering, prompts) is "just" rewriting code. Optimization can target performance and cost.

## The Design

NVIDIA OO Agents: a Python OOP-based agentic framework

- **Generation Methods invoke a code-act loop** — signature (strongly-typed i/o) + docstring (prompt) is the spec
- **Tools/Orchestration/Libraries are just Python** — import libraries, write python tools.
- **SW1/SW3 symmetry** — deterministic Python and LLM reasoning, side by side. Type-safe, async-first, observable
- **Subagents are just classes** – create them and call their methods
- **Agent APIs for Agents** - build APIs for context, history, and documentation

## Setup

One cell to configure everything for the demos.

In [ ]:
import getpass
import os

if "NVIDIA_INTERNAL_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_INTERNAL_API_KEY"] = nvidia_api_key

In [ ]:
from util.presentation_utils import setup, trace

setup()

from unifiedllm.registry import get_llm_client

llm = get_llm_client("aws/anthropic/bedrock-claude-sonnet-4-5-v1")
# llm = get_llm_client("nvidia/nvidia/Nemotron-3-Nano-30B-A3B")
# llm = get_llm_client("nvidia/nvidia/nemotron-super-3-preview")

## Hello, Agent

The simplest possible agent.

An ellipsis body (`...`) signifies a **Generation Method**, meaning **the LLM implements this**.

The method signature + docstring **is** the prompt.

In [ ]:
from nooa import Agent


class Greeter(Agent, llm=llm):
    async def greet(self, name: str) -> str:
        """Write a warm greeting."""
        ...

In [ ]:
trace("01_hello_agent")
agent = Greeter()
# agent_pprint(agent)
result = await agent.greet("Chip Design Applied AI Research Team")
print(result)

That's the whole thing. No decorators, no tool schemas, no prompt files.

## Hello, Agent, Again

The Agent is an object so let's give it state.

In [ ]:
from nooa import Agent
from nooa.agentdoc import pprint


class Greeter(Agent, llm=llm):
    def __init__(self):
        super().__init__()
        self.type = "warm"
        self.times_called = 0

    async def greet(self, name: str) -> str:
        """Increment self.times_called, then write a {self.type} greeting."""
        ...

In [ ]:
trace("01_hello_agent")
agent = Greeter()
pprint(agent)
result = await agent.greet("Chip Design Applied AI Research Team")
print(result)
print(agent.times_called)
agent.type = "sarcastic and a bit mean"
result = await agent.greet("Chip Design Applied AI Research Team")
print(result)
print(agent.times_called)

## Tools are Methods. Types are Types. Libraries are libraries.

Regular Python methods and types are **automatically** visible to the LLM.

- **SW1** (deterministic): normal methods → tools the LLM can call
- **SW3** (generative): ellipsis methods → LLM-implemented

No `@tool` decorator. No JSON schema. No registration. Use your favorite libraries or write new Python libraries.

In [ ]:
from typing import Annotated

from pydantic import BaseModel, Field


class OrderResult(BaseModel):
    can_fulfill: Annotated[
        bool, Field(description="Whether all requested items can be fulfilled within budget")
    ]
    total_cost: Annotated[float, Field(description="Total cost of available items")]
    items_available: Annotated[list[str], Field(description="Items that are in stock")]
    items_unavailable: Annotated[
        list[str], Field(description="Items that are out of stock or not found")
    ]
    reason: Annotated[str, Field(description="Explanation of the fulfillment decision")]


INVENTORY = {
    "apple": {"stock": 50, "price": 0.75},
    "banana": {"stock": 30, "price": 0.50},
    "orange": {"stock": 0, "price": 0.80},  # Out of stock!
    "milk": {"stock": 20, "price": 3.50},
    "bread": {"stock": 15, "price": 2.25},
    "cheese": {"stock": 10, "price": 5.00},
}


class InventoryAgent(Agent, llm=llm):
    """An agent that checks inventory and fulfills orders."""

    # SW1: deterministic tools — the LLM can call these
    def get_stock(self, item: str) -> int:
        """Get current stock for an item."""
        return INVENTORY.get(item.lower(), {}).get("stock", 0)

    def get_price(self, item: str) -> float:
        """Get price for an item."""
        return INVENTORY.get(item.lower(), {}).get("price", 0.0)

    # SW3: LLM-generated — calls the tools above
    async def can_fulfill_order(self, items: list[str], budget: float) -> OrderResult:
        """Check if the order can be fulfilled within budget."""
        ...

In [ ]:
trace("02_inventory_agent")
inv = InventoryAgent()
# pprint(inv)
result = await inv.can_fulfill_order(["apple", "orange", "milk"], budget=10.0)
print(result)

## What does the LLM actually see?

The LLM sees `doc(self)` — auto-generated API documentation of the agent.

In [ ]:
from nooa.agentdoc import doc

print(doc(inv, concise=True))

## Strategies: How Methods Execute

| Strategy | When to use |
|----------|------------|
| **PredictStrategy** | Classification, extraction, single-shot |
| **CodeAct** | Pure code generation |

Per-method via `@strategy(...)`. Composable and extensible.

In [ ]:
from typing import Annotated

from nooa import CodeActStrategy, PredictStrategy, strategy
from nooa.config import CodeActConfig


class BulkClassification(BaseModel):
    classifications: Annotated[list[str], "Sentiment label for each input text"]
    summary: Annotated[str, "Overall pattern across all classifications"]


class AnalysisAgent(Agent, llm=llm):
    """Agent demonstrating different strategies on the same class."""

    # PredictStrategy: fast, single-shot, no tool use
    @strategy(PredictStrategy())
    async def classify_sentiment(self, text: str) -> str:
        """Classify as positive, negative, or neutral. Return only the label."""
        ...

    # CodeActStrategy: multi-step, can call methods, iterate
    @strategy(CodeActStrategy(config=CodeActConfig(max_iterations=10)))
    async def analyze_and_compare(self, texts: list[str]) -> BulkClassification:
        """Classify each text's sentiment using self.classify_sentiment(), then summarize the overall pattern."""
        ...

In [ ]:
trace("03_sentiment_structured")
analyst = AnalysisAgent()

# Single-shot structured output
# agent_pprint(analyst)

sentiment = await analyst.classify_sentiment("This sucks.")
print(f"Sentiment: {sentiment}")

In [ ]:
import time

trace("04_sentiment_codeact")
# CodeAct: iterates, calls classify_sentiment for each text, then summarizes
reviews = [
    "Terrible experience. Broke after one day.",
    "It's okay. Nothing special.",
    "Exceeded all my expectations!",
    "Best agent framework ever!",
    "I'll never write agents any other way.",
]
# agent_pprint(analyst)
t0 = time.time()
summary = await analyst.analyze_and_compare(reviews)
serial_time = time.time() - t0
print(summary)
print(f"\n⏱ Big model, serial: {serial_time:.1f}s")

## Orchestration: From Big-Model Serial to Small-Model Parallel

The previous demo used **one big model** classifying reviews **serially**.

What if we want speed and cost efficiency?

- **Small-model subagents** — each a cheap, fast classifier
- **`asyncio.gather`** — fan out in parallel, collect results
- **Orchestrator** — a big model that delegates and summarizes

Same task. Faster. Cheaper.

In [ ]:
small_llm = get_llm_client("aws/anthropic/claude-haiku-4-5-v1")


class SentimentWorker(Agent, llm=small_llm):
    """A lightweight sentiment classifier powered by a small model."""

    @strategy(PredictStrategy())
    async def classify(self, text: str) -> str:
        """Classify as positive, negative, or neutral. Return only the label."""
        ...


class BulkSentimentAgent(Agent, llm=llm):
    """Orchestrates parallel sentiment analysis using small-model workers."""

    SentimentWorker = SentimentWorker  # expose to generated code

    @strategy(CodeActStrategy(config=CodeActConfig(max_iterations=10)))
    async def analyze_reviews(self, reviews: list[str]) -> BulkClassification:
        """Classify each review in parallel using SentimentWorker instances and asyncio.gather.
        Then summarize the overall pattern."""
        ...

In [ ]:
trace("05_orchestration_parallel")
bulk = BulkSentimentAgent()
pprint(bulk)
t0 = time.time()
result = await bulk.analyze_reviews(reviews)  # same reviews from earlier
parallel_time = time.time() - t0
print(result)
print(f"\n⏱ Small model, parallel: {parallel_time:.1f}s (vs {serial_time:.1f}s serial)")

## "What Is Actually Happening?!"

All of the demos above have been running with **tracing enabled**.

Every LLM call, every generated code block, every tool result — captured.

Let's open the trace viewer and look inside.

### Now let's switch to the trace viewer...

**What to look for:**
- The actual system prompt (context blocks)
- The generated Python code
- Tool call results
- The full method call → strategy → LLM → execution → result flow

## Context Management: An API, Not a Prompt

The agent's context (system prompt) is made of **blocks** — each one a Python expression.

Blocks are inspectable, mutable, and available as an API to both developers and agents.

No prompt templates. No string concatenation. Just `self.context.set()`.

There's a matching API for **event/history management** (`self.events`) — same idea, programmatic control over what the LLM remembers.

### Live: The Context API

What does an agent's context look like from the inside?

## The Two Pillars: Context and History

Everything the LLM sees is controlled through two APIs:

```
                        ┌─────────────────────────────────────┐
                        │          SYSTEM PROMPT              │
                        │  ┌────────────────────────────────┐ │
                        │  │ block: "system_prompt"         │ │  ← identity, instructions
                        │  │ block: "self"                  │ │  ← agent's own API (doc)
                        │  │ block: "context_api"           │ │  ← how to mutate context
                        │  │ block: "notes"  (user-defined) │ │  ← agent can add these
                        │  │ block: "plan"   (user-defined) │ │
                        │  └────────────────────────────────┘ │
                        │  self.context API:                  │
                        │    .set(key, expr=... | value=...)  │
                        │    .get(key) → Block                │
                        │    .remove(key)                     │
                        │    .list_keys()                     │
                        └─────────────────────────────────────┘

                        ┌─────────────────────────────────────┐
                        │           MESSAGES                  │
                        │  ┌────────────────────────────────┐ │
                        │  │ [1] Task("analyze this data")  │ │
                        │  │ [2] LLMOutput(code="...")      │ │
                        │  │ [3] PythonOutput(stdout="...") │ │
                        │  │ [4] LLMOutput(code="...")      │ │
                        │  │ [5] Message("Here are results")│ │
                        │  └────────────────────────────────┘ │
                        │  self.events API:                   │
                        │    .query(type=..., query=...)      │
                        │    .get(tag) → Event                │
                        │    ["tag"] → Event                  │
                        └─────────────────────────────────────┘

**Developers** configure blocks at definition time. **Agents** mutate them at runtime — same API.

## Blocks Render Into Addressable Prompts

Each block is rendered with its **expr** and **metadata** visible to the LLM:

```xml
<system_prompt expr="self.system_prompt()">
  You are an inventory agent that checks stock and fulfills orders.
</system_prompt>

<self expr="doc(self)">
  class InventoryAgent:
    def get_stock(self, item: str) -> int: ...
    def get_price(self, item: str) -> float: ...
    async def can_fulfill_order(self, items: list[str], budget: float) -> OrderResult: ...
</self>

<notes expr="self.context['notes']">
  Found that oranges are out of stock. Adjusting plan.
</notes>
```

The LLM sees **how** each block was produced (`expr`) and **how to mutate** it.

A value block's `expr` is its accessor — the LLM can write `self.context.get('notes').value = "new content"` to update it.

An expression block re-evaluates each turn — `doc(self)` always reflects the latest state.

## Current Status

**Where we are:**
- Internal alpha release in ~2 weeks. Usable now — cleanup on code, prompts, and defining the interface with NAT.
- Very happy to get feedback or beta testers (this notebook is available, for example).

## Next Steps

Mapped to our success metrics:

- **Happy developers** → alpha release in 2w to internal teams

- **E2E optimization** → continue automatic end-to-end optimization of DABStep agents

- **Post-training** → build up expertise to do post-training (bootstrapping)

# Questions?

This notebook is yours — run it, break it, build on it.